## Evaluate a custom Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework. It builds upon [notebook 4](4_Evaluate_Presidio_Analyzer.ipynb), customising the `PresidioAnalyzer` instance for better accuracy. For more on customising Presidio, see the [Presidio Analyzer documentation](https://data-privacy-stack.github.io/presidio/analyzer/).

Steps:
1. Load dataset
2. Dataset statistics
3. Define the custom Presidio Analyzer
4. Run predictions
5. Review and adjust entity mapping
6. Evaluate
7. Results and error analysis

In [27]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator
#!pip install "presidio-analyzer[transformers]"

In [28]:
import json
import re
import warnings
from collections import Counter
from pathlib import Path
from pprint import pprint

warnings.filterwarnings('ignore')

import pandas as pd
from presidio_analyzer import AnalyzerEngine, RecognizerResult
from presidio_analyzer.context_aware_enhancers import LemmaContextAwareEnhancer
from presidio_analyzer.nlp_engine import SlimSpacyNlpEngine
from presidio_analyzer.recognizer_registry import RecognizerRegistryProvider

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import CanonicalMapper
from presidio_evaluator.evaluation import ModelError, Plotter, SpanEvaluator
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import PresidioAnalyzerWrapper

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

%reload_ext autoreload
%autoreload 2
%matplotlib inline


## 1. Load dataset from file

In [29]:
dataset_name = "generated_size_1500_date_October_23_2026.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))

print(len(dataset))

tokenizing input: 100%|██████████| 1500/1500 [00:06<00:00, 248.35it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [30]:
def get_entity_counts(dataset: list[InputSample]) -> dict:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

## 2. Simple dataset statistics

In [31]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print(
    "\nMin and max number of tokens in dataset: "
    f"Min: {min([len(sample.tokens) for sample in dataset])}, "
    f"Max: {max([len(sample.tokens) for sample in dataset])}"
)

print(
    f"Min and max sentence length in dataset: "
    f"Min: {min([len(sample.full_text) for sample in dataset])}, "
    f"Max: {max([len(sample.full_text) for sample in dataset])}"
)

Count per entity:
[('O', 19406), ('STREET_ADDRESS', 1982), ('DATE_TIME', 1177), ('PERSON', 1052),
 ('ORGANIZATION', 604), ('PHONE_NUMBER', 208), ('EMAIL_ADDRESS', 187),
 ('UK_DRIVING_LICENCE', 128), ('TITLE', 128), ('CREDIT_CARD', 67), ('GPE', 42),
 ('IBAN_CODE', 34), ('AGE', 21)]

Min and max number of tokens in dataset: Min: 6, Max: 32
Min and max sentence length in dataset: Min: 30, Max: 142


## 3. Define the AnalyzerEngine object 
In this case, we customize the AnalyzerEngine to use a different NER model, some custom recognizers and the context aware enhancer.

### 3.1 Set up the NlpEngine
The NLP engine uses spaCy for text processing (lemmas, tokens, etc.). The Hugging Face model is added as a direct recognizer (HuggingFaceNerRecognizer) instead of being part of the NLP engine. This approach gives better control over model predictions.

In [32]:
# Defin a slim Nlp Engine (we're using a Presidio recognizer instead of a Presidio NLP Engine)
nlp_engine = SlimSpacyNlpEngine()
nlp_engine.load()

### 3.2 Set up the relevant recognizers
The recognizer registry is defined in [recognizers.yaml](../recognizers.yaml), tuned on this dataset and focused on UK PII (no US-specific recognizers):
- The Hugging Face NER recognizer (`OpenMed-PII-SuperClinical-Large-434M-v1`) with its label mapping to Presidio entities.
- Predefined pattern recognizers, including `UkDrivingLicenceRecognizer` (disabled in Presidio's defaults).
- Custom pattern recognizers for UK NINO, passport numbers and postcodes, card numbers, weekdays and phone numbers in international formats. Some only fire when context (e.g. "passport") lifts their score above the threshold.
- `PhoneRecognizer` restricted to GB numbers; `SpacyRecognizer` is disabled.

Each entry in the YAML file notes why it is there.

In [33]:
recognizers_conf = Path(Path.cwd().parent, "recognizers.yaml")
registry = RecognizerRegistryProvider(
    conf_file=recognizers_conf, nlp_engine=nlp_engine
).create_recognizer_registry()


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

### 3.3 Configure the context mechanism
Configure the `LemmaContextAawareEnhancer` which uses surrounding words to increase confidence in detection

In [34]:
# Set up the context aware enhancer
context_enhancer = LemmaContextAwareEnhancer(
    context_prefix_count=10, context_suffix_count=10
)

### 3.4 Post-process the analyzer results
`PostProcessingAnalyzerEngine` extends `AnalyzerEngine.analyze` with rules that align the model's spans with how this dataset is annotated:
- Trim template artefacts (literal `\n`, `}}`) from span edges, and cut phone/email/URL/address spans at literal `\n` line breaks.
- Let structured-ID pattern matches (NINO, passport, licence, card) override overlapping NER spans.
- Union overlapping organisation spans into a single `ORGANIZATION`.
- Merge street/city/postcode/country chains into one `STREET_ADDRESS`; standalone city/country spans become `GPE`, or `NRP` for demonyms such as "Australian".
- Drop occupations mentioned in prose (only standalone job-title fields are annotated as `TITLE`) and short all-caps "names" such as "NINO".

Several of these rules mirror quirks of the synthetic data generator, so revisit them when evaluating on a different dataset.

In [35]:
# Recognizers (by name in recognizers.yaml) whose matches override overlapping NER spans
PRIORITY_RECOGNIZERS = {
    "CardNumberContextRecognizer",
    "UkNinoPatternRecognizer",
    "UkDrivingLicenceRecognizer",
    "UkPassportNumberRecognizer",
}

LINE_END = re.compile(r"^\s*(?:\\n|\n|$)")
# Nationality adjectives the NER model labels as countries ('Australian', 'Danish', 'Czech')
DEMONYM = re.compile(
    r"^(?!(?:Japan|Iran|Oman|Sudan|Taiwan|Bhutan|Azerbaijan|Vatican)$)(?!\w+stan$)"
    r"(?:[A-Z]\w*(?:ian|ean|ese|ish|an|ic)|German|Czech|Greek|Swiss|Dutch|French|Thai)$"
)
NAME_PARTS = {"PERSON", "FIRST_NAME", "LAST_NAME"}
ORG_TYPES = {"ORGANIZATION", "COMPANY"}
ADDRESS_PARTS = {"STREET_ADDRESS", "CITY", "COUNTRY", "COUNTY", "STATE", "ZIP_CODE"}
ADDRESS_TAIL = {"ZIP_CODE", "COUNTRY", "COUNTY", "STATE"}
GPE_PARTS = {"CITY", "COUNTRY", "COUNTY", "STATE"}
# Text allowed between two address parts. Literal "\n" sequences mark template
# layouts where the dataset annotates each address part separately.
ADDRESS_GAP = re.compile(r"^(?:[\s,()\-/]|nan)*$")  # "nan": empty field in the generator


# Span edges that are never part of an entity (literal "\n" and "}}" come from the
# dataset's templates)
EDGE_JUNK = re.compile(r"^(?:\s|\\n|\}\}|,)+|(?:\s|\\n|\}\}|,)+$")


def trim_span(text, r):
    value = text[r.start : r.end]
    lead = EDGE_JUNK.match(value)
    if lead and lead.end() < len(value):
        r.start += lead.end()
        value = value[lead.end() :]
    trail = EDGE_JUNK.search(value)
    if trail and trail.start() > 0:
        r.end -= len(value) - trail.start()
    return r


LITERAL_NEWLINE = re.compile(r"(?:\\n)+")
SEGMENT_TEST = {
    "PHONE_NUMBER": lambda seg: sum(c.isdigit() for c in seg) >= 4,
    "EMAIL_ADDRESS": lambda seg: "@" in seg,
    "DOMAIN_NAME": lambda seg: "." in seg,
}


def split_at_literal_newline(text, r):
    """Shrink a span to the literal-"\\n"-delimited segment that holds the entity.

    The dataset's templates contain literal "\\n" sequences that tokenizers do not
    split on, so model spans run across neighbouring fields.
    """
    test = SEGMENT_TEST.get(r.entity_type)
    if test is None:
        return r
    value = text[r.start : r.end]
    pos = 0
    for part in LITERAL_NEWLINE.split(value):
        offset = value.index(part, pos)
        pos = offset + len(part)
        if part.strip() and test(part):
            return trim_span(text, RecognizerResult(r.entity_type, r.start + offset, r.start + pos, r.score,
                                                    r.analysis_explanation, r.recognition_metadata))
    return r


def split_address_lines(text, r):
    """Template addresses put one part per (literal "\\n") line and annotate each line."""
    value = text[r.start : r.end]
    if r.entity_type != "STREET_ADDRESS" or not LITERAL_NEWLINE.search(value):
        return [r]
    parts, pos = [], 0
    for part in LITERAL_NEWLINE.split(value):
        offset = value.index(part, pos)
        pos = offset + len(part)
        if part.strip(" }"):
            parts.append(trim_span(text, RecognizerResult(r.entity_type, r.start + offset, r.start + pos, r.score,
                                                          r.analysis_explanation, r.recognition_metadata)))
    return parts


class PostProcessingAnalyzerEngine(AnalyzerEngine):
    """Merge chains of adjacent address parts into a single STREET_ADDRESS.

    The NER model splits '87 Stevenson Forest, Lydiamouth, CB9 2ES' into
    street/city/postcode spans, while the dataset labels the full address.
    City/country/state spans outside an address chain become GPE.
    """

    def analyze(self, text, *args, **kwargs):
        results = [trim_span(text, r) for r in super().analyze(text, *args, **kwargs)]
        results = [split_at_literal_newline(text, r) for r in results]
        results = [part for r in results for part in split_address_lines(text, r)]
        results = self._apply_priority(results)
        results = self._union_orgs(results)
        # Short all-caps tokens ('NINO') are acronyms, not names
        results = [
            r for r in results
            if not (r.entity_type in NAME_PARTS and text[r.start : r.end].isupper() and r.end - r.start <= 4)
        ]
        # Job titles are annotated as standalone fields (ending a line), not as occupations in prose
        results = [r for r in results if not self._is_prose_occupation(text, r)]
        parts = sorted((r for r in results if r.entity_type in ADDRESS_PARTS), key=lambda r: r.start)
        others = [r for r in results if r.entity_type not in ADDRESS_PARTS]

        chains, chain = [], []
        for r in parts:
            if chain and (r.start <= chain[-1].end or ADDRESS_GAP.match(text[chain[-1].end : r.start])):
                chain.append(r)
            else:
                if chain:
                    chains.append(chain)
                chain = [r]
        if chain:
            chains.append(chain)

        merged = []
        for chain in chains:
            types = {r.entity_type for r in chain}
            # 'street, city' alone is annotated as separate parts; full addresses end in a
            # postcode, region or country
            if len(chain) > 1 and "STREET_ADDRESS" in types and types & ADDRESS_TAIL:
                merged.append(self._merge(chain, "STREET_ADDRESS"))
            else:
                for r in chain:
                    if r.entity_type in GPE_PARTS:
                        r.entity_type = "NRP" if DEMONYM.match(text[r.start : r.end]) else "GPE"
                    merged.append(r)
        return others + merged

    def _union_orgs(self, results):
        """Overlapping organisation spans (NER + suffix pattern) become their union."""
        orgs = sorted((r for r in results if r.entity_type in ORG_TYPES), key=lambda r: r.start)
        merged = []
        for r in orgs:
            if merged and r.start < merged[-1][-1].end:
                merged[-1].append(r)
            else:
                merged.append([r])
        return [r for r in results if r.entity_type not in ORG_TYPES] + [
            self._merge(group, "ORGANIZATION") for group in merged
        ]

    @staticmethod
    def _is_prose_occupation(text, r):
        if r.entity_type != "TITLE" or not text[r.start].islower():
            return False
        value = text[r.start : r.end]
        return "\\n" not in value and "\n" not in value and not LINE_END.match(text[r.end :])

    @staticmethod
    def _apply_priority(results):
        """Structured-ID pattern matches override any overlapping span from other recognizers."""
        strong = [r for r in results if r.recognition_metadata.get("recognizer_name") in PRIORITY_RECOGNIZERS]
        if not strong:
            return results
        return strong + [
            r
            for r in results
            if r.recognition_metadata.get("recognizer_name") not in PRIORITY_RECOGNIZERS
            and not any(r.start < s.end and s.start < r.end for s in strong)
        ]

    @staticmethod
    def _merge(chain, entity_type):
        return RecognizerResult(
            entity_type=entity_type,
            start=min(r.start for r in chain),
            end=max(r.end for r in chain),
            score=max(r.score for r in chain),
            analysis_explanation=chain[0].analysis_explanation,
            recognition_metadata=chain[0].recognition_metadata,
        )


### 3.5 Create the AnalyzerEngine object

In [36]:
# Set up the engine and other PII recognizers
analyzer_engine = PostProcessingAnalyzerEngine(
    nlp_engine=nlp_engine,
    context_aware_enhancer=context_enhancer,
    registry=registry,
    default_score_threshold=0.3,
)

# analyzer_engine = PostProcessingAnalyzerEngine()

pprint("Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint(
    [
        rec.name
        for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)
    ],
    compact=True,
)

print("\nLoaded Context Aware Enhancer:")
print(analyzer_engine.context_aware_enhancer.__class__.__name__)
pprint(json.dumps(analyzer_engine.context_aware_enhancer.__dict__), compact=True)


'Supported entities for English:'
['EMPLOYMENT_STATUS', 'TITLE', 'COORDINATE', 'LICENSE_PLATE', 'COMPANY',
 'ZIP_CODE', 'GENDER', 'PROFESSIONAL_LICENSE', 'NRP', 'CVV', 'MAC_ADDRESS',
 'ACCOUNT_NUMBER', 'UK_DRIVING_LICENCE', 'IBAN_CODE', 'PHONE_NUMBER',
 'USER_NAME', 'UK_PASSPORT', 'BIOMETRIC_IDENTIFIER', 'PASSWORD', 'ID',
 'CREDIT_CARD', 'POLITICAL_VIEW', 'LAST_NAME', 'SWIFT_CODE', 'EDUCATION_LEVEL',
 'STREET_ADDRESS', 'MEDICAL_LICENSE', 'UK_NINO', 'CITY', 'DATE_TIME',
 'FIRST_NAME', 'AGE', 'IP_ADDRESS', 'SEXUALITY', 'COUNTRY', 'EMAIL_ADDRESS',
 'DOMAIN_NAME', 'BLOOD_TYPE', 'STATE', 'COUNTY', 'RELIGIOUS_BELIEF',
 'BANK_ROUTING_NUMBER']

Loaded recognizers for English:
['HuggingFaceNerRecognizer', 'CreditCardRecognizer', 'DateRecognizer',
 'EmailRecognizer', 'IbanRecognizer', 'IpRecognizer', 'MacAddressRecognizer',
 'PhoneRecognizer', 'UkDrivingLicenceRecognizer', 'TitlesRecognizer',
 'YearsRecognizer', 'WeekdayRecognizer', 'GenericPhoneRecognizer',
 'UkPostcodePatternRecognizer', 'UkNi

In [37]:
# Test Analyzer
text = "Yesterday in Mt. Sinai AP: Dana Silver, 79 years old female was complaining of stomach pain. Her ID is 154555"
res = analyzer_engine.analyze(text=text, language="en", return_decision_process=True)
for result in res:
    print(f"Entity: {result.entity_type}, Text: {text[result.start : result.end]}")

Entity: FIRST_NAME, Text: Dana
Entity: LAST_NAME, Text: Silver
Entity: AGE, Text: 79
Entity: ID, Text: 154555
Entity: GENDER, Text: female
Entity: GPE, Text: Mt. Sinai


In [38]:
# Entities annotated in the dataset (plus the address parts the post-processor needs)
entities_to_keep = [
    "PERSON", "FIRST_NAME", "LAST_NAME", "TITLE", "AGE", "NRP", "DATE_TIME",
    "EMAIL_ADDRESS", "PHONE_NUMBER", "DOMAIN_NAME", "IP_ADDRESS",
    "CREDIT_CARD", "IBAN_CODE",
    "UK_NINO", "UK_PASSPORT", "UK_DRIVING_LICENCE",
    "STREET_ADDRESS", "CITY", "COUNTRY", "COUNTY", "STATE", "ZIP_CODE", "GPE",
    "ORGANIZATION", "COMPANY",
]

# Wrap the analyzer
wrapped_analyzer = PresidioAnalyzerWrapper(
    analyzer_engine=analyzer_engine,
    entities_to_keep=entities_to_keep,
    score_threshold=analyzer_engine.default_score_threshold,
    language="en",
)

## 4. Run predictions

Run the model on the full dataset. Returns a 5-column DataFrame (`sentence_id`, `token`, `annotation`, `prediction`, `start_indices`).

In [39]:
results_df = wrapped_analyzer.predict_dataset(dataset)
results_df.head()

,sentence_id,token,annotation,prediction,start_indices,annotation_span_id,prediction_span_id
0,0,My,O,O,0,None,None
1,0,employer,O,O,3,None,None
2,0,is,O,O,12,None,None
3,0,Synchronoss,ORGANIZATION,ORGANIZATION,15,1,1
4,0,Technologies,ORGANIZATION,ORGANIZATION,27,1,1


## 5. Review entity mapping

- The entity mapping process auto-resolves labels using a hierarchical entity tree.
The `CanonicalMapper` class analyzes the results (predictions and annotations), and checks if there are any gaps towards the evaluation process.

- If you call `render_html()` to inspect issues, you'd see if there are any gaps that prevent you from completing the evaluation successfully. Make sure you address all the issues by using:
  -  `mapper.map({"LABEL": "CANONICAL"})` to remap or 
  -  `mapper.map({"LABEL": None})` to suppress.

- For more information about entity mapping, see [Notebook 6 - Entity Mapping](6_Interactive_Entity_Mapping.ipynb).

In [40]:
# Analyze the results DataFrame.
# min_collision_count suppresses COLLISION_CROSS_BRANCH warnings with fewer
# cross-branch token co-occurrences than the threshold (rare label overlaps).
mapper = CanonicalMapper()
mapper.analyze(results_df, min_collision_count=5)
mapper.render_html()

Label,Resolved as,Tokens
STREET_ADDRESS,ADDRESS,1982
DATE_TIME,DATE_TIME,1177
PERSON,PERSON,1052
ORGANIZATION,ORGANIZATION,604
PHONE_NUMBER,PHONE_NUMBER,208
EMAIL_ADDRESS,EMAIL_ADDRESS,187
TITLE,TITLE,128
CREDIT_CARD,FINANCIAL,67
GPE,GPE,42
IBAN_CODE,FINANCIAL,34


In [41]:
# Suppress all those entities that the model detects but the dataset doesn't have annotations for (prediction-only labels)
mapper.suppress_prediction_only()

In [42]:
# Resolve blocking issues. Adjust mappings for your model's labels.
# Unresolved and prediction-only labels are suppressed here — remap if appropriate.
# Mixed gold depths need no decision: each prediction is projected to the
# deepest annotated ancestor of its own label.
resolutions = {}
for issue in mapper.get_issues():
    if issue.type.value in ("unresolved", "prediction_only"):
        resolutions.update(dict.fromkeys(issue.labels))

if resolutions:
    print("Applying explicit mappings before evaluation:", resolutions)
    mapper.map(resolutions)

mapped_results = mapper.get_mapped_results_dataframe()
mapper.render_html()

Label,Resolved as,Tokens
STREET_ADDRESS,ADDRESS,1982
DATE_TIME,DATE_TIME,1177
PERSON,PERSON,1052
ORGANIZATION,ORGANIZATION,604
PHONE_NUMBER,PHONE_NUMBER,208
EMAIL_ADDRESS,EMAIL_ADDRESS,187
TITLE,TITLE,128
CREDIT_CARD,FINANCIAL,67
GPE,GPE,42
IBAN_CODE,FINANCIAL,34


## 6. Evaluate

In [43]:
# Set up the experiment tracker and log model + dataset params
experiment = get_experiment_tracker()
params = {"dataset_name": dataset_name, "model_name": wrapped_analyzer.name}
params.update(wrapped_analyzer.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)
experiment.log_parameter("entity_mappings", json.dumps(mapper.get_mapping()))

evaluator = SpanEvaluator(iou_threshold=0.75)
f_beta = 2  # F-beta parameter used throughout evaluation

skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]


In [44]:
hierarchical_results = evaluator.calculate_hierarchical_scores(
    mapped_results,
    beta=f_beta
    )
binary_results = hierarchical_results["binary"]
branch_results = hierarchical_results["branch"]
detailed_results = hierarchical_results["detailed"]

# Log detailed-level metrics to experiment tracker
experiment.log_metrics(detailed_results.to_log())
entities, confmatrix = detailed_results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

# A. Binary PII/O metrics
print("Level: binary (PII vs O):")
pprint({
    "Precision": round(binary_results.pii_precision, 3),
    "Recall":    round(binary_results.pii_recall, 3),
    f"F{f_beta}":  round(binary_results.pii_f, 3),
})

saving experiment data to /Users/oleksii.nikitchuk/Documents/carmoola/presidio-research/notebooks/experiment_20261023-124446.json
Level: binary (PII vs O):
{'F2': 0.975, 'Precision': 0.985, 'Recall': 0.973}


In [45]:
# B. Branch-level per-entity scores (PERSON, LOCATION, DATE_TIME, …)
branch_plotter = Plotter(
    results=branch_results,
    model_name=wrapped_analyzer.name,
    display_mode="interactive", 
    beta=f_beta,
)
branch_plotter.plot_scores(include_pii_aggregate=False, annotation_entities_only=True)

In [46]:
# C. Detailed-level per-entity scores (NAME, GPE, PHONE_NUMBER, …)
detailed_plotter = Plotter(
    results=detailed_results,
    model_name=wrapped_analyzer.name,
    display_mode="interactive",
    beta=f_beta,
)
detailed_plotter.plot_scores(include_pii_aggregate=False, annotation_entities_only=True)

## 7. Confusion matrix

In [47]:
# D. Branch-level confusion matrix (only branch entities that appear in the data)
entities, confmatrix = branch_results.to_confusion_matrix()
branch_plotter.plot_confusion_matrix(entities=entities, confmatrix=confmatrix)

## 8. Error analysis

The `ModelError` class groups similar errors together, finding all the false positive and false negative predictions where the model's behavior is consistent. 

This can help in detecting patterns in the model's performance, such as:
- Consistent labeling of specific words or patterns  
- Systematic overprediction or underprediction

**Note:** All errors are displayed using the **mapped model entity types** (e.g., `LOCATION`, `PERSON`) rather than the original dataset entity types (e.g., `STREET_ADDRESS`, `GPE`). This ensures consistency with how the model was evaluated.

In [48]:
branch_plotter.plot_most_common_tokens()

### 8a. False positives
#### Most common false positive tokens:

In [53]:
ModelError.most_common_fp_tokens(branch_results.model_errors)

Most common false positive tokens:
[('nv', 2),
 ('administrator', 2),
 ('bahri', 2),
 ('wigan', 2),
 ('tagnifi', 2),
 ('esri', 2),
 ('nucivic', 2),
 ('colony c 7.15 srs cum red prep prf', 2),
 ('sa', 2),
 ('etfs comm/1x', 1)]
---------------
Example sentence with each FP token:
	- NV (`nv` pred as LOCATION)
	- Administrator (`administrator` pred as PERSON)
	- BAHRI (`bahri` pred as PERSON)
	- Wigan (`wigan` pred as DEMOGRAPHIC)
	- Tagnifi (`tagnifi` pred as LOCATION)
	- Esri (`esri` pred as LOCATION)
	- Nucivic (`nucivic` pred as PERSON)
	- COLONY C 7.15 SRS I CUM RED PREP PRF (`colony c 7.15 srs cum red prep prf` pred as LOCATION)
	- SA (`sa` pred as LOCATION)
	- ETFS COMM/1X (`etfs comm/1x` pred as ORGANIZATION)


[('nv', 2),
 ('administrator', 2),
 ('bahri', 2),
 ('wigan', 2),
 ('tagnifi', 2),
 ('esri', 2),
 ('nucivic', 2),
 ('colony c 7.15 srs cum red prep prf', 2),
 ('sa', 2),
 ('etfs comm/1x', 1)]

#### More FP analysis

In [59]:
fps_df = ModelError.get_fps_dataframe(branch_results.model_errors, entity="PERSON")
fps_df[["full_text", "token", "annotation", "prediction", ]].head(20)

,full_text,token,annotation,prediction
0,Administrator,administrator,O,PERSON
1,BAHRI,bahri,O,PERSON
2,SUJITH,sujith,O,PERSON
3,Accountant,accountant,O,PERSON
4,Administrator,administrator,O,PERSON
5,KINKAI,kinkai,O,PERSON
6,Scientist,scientist,O,PERSON
7,ronald,ronald,O,PERSON
8,Horticulturist,horticulturist,O,PERSON
9,Crawford,crawford,O,PERSON


### 8b. False negatives (FN)

#### Most common false negative examples + a few samples with FN

In [51]:
ModelError.most_common_fn_tokens(branch_results.model_errors, n=15)

Most common false negative tokens:
[('5', 2),
 ('amit bahri', 2),
 ('sqli sa', 2),
 ('wigan', 2),
 ('etfs oil etfs wti oil 2yr', 2),
 ('tagnifi', 2),
 ('esri', 2),
 ('nucivic', 2),
 ('colony c 7.15 srs cum red prep prf', 2),
 ('etfs comm/1x daily short agricu', 1),
 ('2.2', 1),
 ('national grid plc', 1),
 ('frst ds rep 0.25 srs non cum prf', 1),
 ('credit enhanced corts trust aon', 1),
 ('jupiter smaller companies plc', 1)]
---------------
Example sentence with each FN token:
	- 5 (`5` annotated as DEMOGRAPHIC)
	- AMIT BAHRI (`amit bahri` annotated as PERSON)
	- SQLI SA (`sqli sa` annotated as ORGANIZATION)
	- Wigan (`wigan` annotated as LOCATION)
	- ETFS OIL / ETFS WTI OIL 2YR (`etfs oil etfs wti oil 2yr` annotated as ORGANIZATION)
	- Tagnifi (`tagnifi` annotated as ORGANIZATION)
	- Esri (`esri` annotated as ORGANIZATION)
	- Nucivic (`nucivic` annotated as ORGANIZATION)
	- COLONY C 7.15 SRS I CUM RED PREP PRF (`colony c 7.15 srs cum red prep prf` annotated as ORGANIZATION)
	- ETFS COM

[('5', 2),
 ('amit bahri', 2),
 ('sqli sa', 2),
 ('wigan', 2),
 ('etfs oil etfs wti oil 2yr', 2),
 ('tagnifi', 2),
 ('esri', 2),
 ('nucivic', 2),
 ('colony c 7.15 srs cum red prep prf', 2),
 ('etfs comm/1x daily short agricu', 1),
 ('2.2', 1),
 ('national grid plc', 1),
 ('frst ds rep 0.25 srs non cum prf', 1),
 ('credit enhanced corts trust aon', 1),
 ('jupiter smaller companies plc', 1)]

#### More FN analysis

In [1]:
fns_df = ModelError.get_fns_dataframe(branch_results.model_errors, entity="LOCATION")
fns_df.head(200)

NameError: name 'ModelError' is not defined